In [13]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Training Run Analysis

Load a training run and visualize global trends across episodes.  
For per‑turn interactive exploration, launch `turn_dashboard.py` instead.

In [14]:
from pathlib import Path
from visu import (
    find_run_dir, load_episode_summaries, load_turn_summaries,
    plot_win_rate, plot_rewards, plot_episode_length,
    plot_cards_played, plot_action_economy,
    plot_episode_turns, plot_last_turn_panel,
    summary_stats,
)

RUN_NAME = "big_run6"
run_dir = find_run_dir(RUN_NAME)
print(f"Run directory: {run_dir}")

Run directory: c:\Users\valen\Desktop\code\swu\forceteki\python_rl\runs\big_run6


In [15]:
ep_df = load_episode_summaries(run_dir)

# turn_summaries.csv only exists for runs recorded with updated train.py
try:
    turn_df = load_turn_summaries(run_dir, ep_df)
    print(f"Turn rows: {len(turn_df)}  (turns {int(turn_df['turn_number'].min())}–{int(turn_df['turn_number'].max())})")
    HAS_TURNS = True
except FileNotFoundError:
    turn_df = None
    HAS_TURNS = False
    print("(no turn_summaries.csv — turn‑level plots skipped)")

print(f"Episodes: {len(ep_df)}")
print(f"Columns: {list(ep_df.columns)}")

Turn rows: 20577  (turns 1–65)
Episodes: 1133
Columns: ['run_id', 'episode', 'agent_turns', 'opponent_turns', 'agent_rewards', 'opponent_rewards', 'total_rewards', 'total_reward_steps', 'valid_actions_sum', 'valid_actions_count', 'agent_valid_actions_sum', 'agent_valid_actions_count', 'agent_ready_resources_sum', 'agent_credits_sum', 'agent_board_power_sum', 'agent_board_hp_sum', 'agent_board_damage_sum', 'agent_unit_count_sum', 'agent_exhausted_sum', 'agent_base_hp_sum', 'agent_leader_hp_sum', 'opp_board_power_sum', 'opp_board_hp_sum', 'opp_board_damage_sum', 'opp_unit_count_sum', 'opp_exhausted_sum', 'opp_base_hp_sum', 'opp_leader_hp_sum', 'agent_hand_sum', 'opp_hand_sum', 'agent_max_valid_actions', 'final_phase', 'winner', 'regroup_segments', 'regroup_segment_count', 'regroup_action_total', 'regroup_card_action_total', 'regroup_agent_action_total', 'regroup_opponent_action_total', 'cards_played', 'agent_cards_played', 'steps', 'agent_reward_per_turn', 'opponent_reward_per_turn', 'av

## Run Summary

In [16]:
stats = summary_stats(ep_df)
for k, v in stats.items():
    print(f"  {k}: {v}")

  n_episodes: 1133
  win_rate: 0.6116504854368932
  avg_agent_reward: -5.125743094963355
  avg_reward_margin: -2.973064049870518
  avg_steps: 139.6472148541114
  avg_turns: 73.71883289124668
  best_margin_ep: 394
  best_margin: 13.708333333333334
  worst_margin_ep: 11
  worst_margin: -52.99440760649743


## Global Trends

In [18]:
WINDOW = 20
plot_win_rate(ep_df, WINDOW).show()

In [19]:
plot_rewards(ep_df, WINDOW).show()

In [26]:
plot_episode_length(ep_df, WINDOW).show()

In [21]:
fig = plot_cards_played(ep_df, WINDOW)
if fig:
    fig.show()
else:
    print("cards_played column not found — re‑run with updated train.py")

In [22]:
plot_action_economy(ep_df, WINDOW).show()

## End‑of‑Game Snapshot Across Training

In [23]:
if HAS_TURNS:
    plot_last_turn_panel(turn_df, window=10).show()
else:
    print("turn_summaries.csv not available for this run")

## Single‑Episode Turn Breakdown

Change `EPISODE` below to inspect a specific episode.

In [24]:
if HAS_TURNS:
    EPISODE = int(ep_df["episode"].iloc[-1])
    try:
        plot_episode_turns(turn_df, EPISODE).show()
    except ValueError as e:
        print(e)
else:
    print("turn_summaries.csv not available for this run")

## Best & Worst Episodes

Loaded from `best_worst_episodes.json` (written by train.py at end of run).

In [25]:
import json
bw_path = run_dir / "best_worst_episodes.json"
if bw_path.exists():
    bw = json.loads(bw_path.read_text())
    print("Best episodes by reward margin")
    for e in bw.get("best_episodes_by_margin", []):
        print(f"  ep {e['episode']:>4}  margin={e['reward_margin']:+.2f}  "
              f"agent={e['agent_rewards']:.2f}  opp={e['opponent_rewards']:.2f}  "
              f"steps={e['steps']}  winner={e['winner']}")
    print("Worst episodes by reward margin")
    for e in bw.get("worst_episodes_by_margin", []):
        print(f"  ep {e['episode']:>4}  margin={e['reward_margin']:+.2f}  "
              f"agent={e['agent_rewards']:.2f}  opp={e['opponent_rewards']:.2f}  "
              f"steps={e['steps']}  winner={e['winner']}")
else:
    print(f"best_worst_episodes.json not found in {run_dir}")

Best episodes by reward margin
  ep  394  margin=+13.71  agent=1.85  opp=-11.85  steps=97  winner=unresolved
  ep  440  margin=+13.70  agent=1.11  opp=-12.59  steps=95  winner=unresolved
  ep  525  margin=+12.96  agent=1.57  opp=-11.39  steps=131  winner=unresolved
  ep  603  margin=+12.75  agent=1.81  opp=-10.94  steps=109  winner=unresolved
  ep  853  margin=+11.97  agent=0.63  opp=-11.34  steps=110  winner=unresolved
Worst episodes by reward margin
  ep   83  margin=-38.33  agent=-40.09  opp=-1.76  steps=281  winner=agent
  ep  153  margin=-36.33  agent=-37.34  opp=-1.01  steps=350  winner=unresolved
  ep   82  margin=-29.80  agent=-31.69  opp=-1.88  steps=287  winner=opponent
  ep  101  margin=-26.90  agent=-30.23  opp=-3.32  steps=347  winner=agent
  ep  145  margin=-26.03  agent=-29.74  opp=-3.70  steps=244  winner=opponent
